#  Multi-Omics Autoencoder Representation Learning

## Objective

The goal of this notebook is to learn a compact representation of cancer cell lines from multi-omics data.

The GraphTCDR framework uses Gene Expression, Copy Number Variation (CNV), and miRNA features to represent cancer cell lines. These three omics datasets contain a very large number of features:

- Gene Expression: 20,404 features
- CNV: 22,582 features
- miRNA: 734 features

Together, they produce:

**20,404 + 22,582 + 734 = 43,720 features per cell line.**

Directly using all 43,720 features would create a very high-dimensional representation. Therefore, an Autoencoder is used to learn a lower-dimensional latent representation.

In this project, the target representation is a **512-dimensional embedding for each cell line**.

The learned representation will later be used as the cell-line representation for downstream drug-response prediction models.

## Important methodological note

The original GraphTCDR implementation provides an Autoencoder architecture for learning the cell-line representation. This notebook will first verify our downloaded input data and then implement the architecture based on the reference implementation.

Any project-specific changes from the reference implementation will be explicitly documented rather than silently treated as an exact reproduction.

Verify Multi-Omics Input Files

In [6]:
import os
import torch

PROJECT = r"D:\Pancreatic_GraphTCDR"
RAW = os.path.join(PROJECT, "data", "raw")

files = {
    "Gene Expression": "GeneExpressionFeature402.pt",
    "CNV": "CNVFeature402.pt",
    "miRNA": "miRNAFeature402.pt"
}

features = {}

for feature_name, filename in files.items():

    path = os.path.join(RAW, filename)

    print(f"\nChecking {feature_name}")
    print(f"Path: {path}")

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{feature_name} file was not found:\n{path}"
        )

    tensor = torch.load(path, map_location="cpu")
    tensor = torch.as_tensor(tensor).float()

    features[feature_name] = tensor

    print(f"Shape: {tuple(tensor.shape)}")
    print(f"Minimum: {tensor.min().item():.6f}")
    print(f"Maximum: {tensor.max().item():.6f}")
    print(f"NaN values: {torch.isnan(tensor).sum().item():,}")
    print(f"Inf values: {torch.isinf(tensor).sum().item():,}")


print("\n" + "=" * 70)
print("EXPECTED SHAPES")
print("=" * 70)

expected_shapes = {
    "Gene Expression": (402, 20404),
    "CNV": (402, 22582),
    "miRNA": (402, 734)
}

for name, expected_shape in expected_shapes.items():

    actual_shape = tuple(features[name].shape)

    print(
        f"{name}: "
        f"expected={expected_shape}, "
        f"actual={actual_shape}"
    )

    if actual_shape != expected_shape:
        raise ValueError(
            f"{name} shape mismatch: "
            f"expected {expected_shape}, "
            f"found {actual_shape}"
        )

print("\nInput-file verification completed.")


Checking Gene Expression
Path: D:\Pancreatic_GraphTCDR\data\raw\GeneExpressionFeature402.pt
Shape: (402, 20404)
Minimum: 0.000000
Maximum: 1.000000
NaN values: 0
Inf values: 0

Checking CNV
Path: D:\Pancreatic_GraphTCDR\data\raw\CNVFeature402.pt
Shape: (402, 22582)
Minimum: 0.000000
Maximum: 1.000000
NaN values: 0
Inf values: 0

Checking miRNA
Path: D:\Pancreatic_GraphTCDR\data\raw\miRNAFeature402.pt
Shape: (402, 734)
Minimum: 0.000000
Maximum: 1.000000
NaN values: 0
Inf values: 0

EXPECTED SHAPES
Gene Expression: expected=(402, 20404), actual=(402, 20404)
CNV: expected=(402, 22582), actual=(402, 22582)
miRNA: expected=(402, 734), actual=(402, 734)

Input-file verification completed.


 Concatenate Multi-Omics Features

In [7]:
X = torch.cat(
    [
        features["Gene Expression"],
        features["CNV"],
        features["miRNA"]
    ],
    dim=1
)

print("=" * 70)
print("CONCATENATED MULTI-OMICS MATRIX")
print("=" * 70)

print(f"Shape: {tuple(X.shape)}")

print(f"\nExpected shape: (402, 43720)")
print(f"Actual shape:   {tuple(X.shape)}")

print(f"\nMinimum value: {X.min().item():.6f}")
print(f"Maximum value: {X.max().item():.6f}")

print(f"\nNaN values: {torch.isnan(X).sum().item():,}")
print(f"Inf values: {torch.isinf(X).sum().item():,}")

expected_features = 20404 + 22582 + 734

if X.shape != (402, expected_features):
    raise ValueError(
        f"Unexpected concatenated shape: {tuple(X.shape)}"
    )

if torch.isnan(X).any():
    raise ValueError("Concatenated matrix contains NaN values.")

if torch.isinf(X).any():
    raise ValueError("Concatenated matrix contains infinite values.")

print("\n" + "=" * 70)
print("VERIFICATION PASSED")
print("=" * 70)

print(
    f"Each of the 402 cell lines is now represented by "
    f"{X.shape[1]:,} multi-omics features."
)

CONCATENATED MULTI-OMICS MATRIX
Shape: (402, 43720)

Expected shape: (402, 43720)
Actual shape:   (402, 43720)

Minimum value: 0.000000
Maximum value: 1.000000

NaN values: 0
Inf values: 0

VERIFICATION PASSED
Each of the 402 cell lines is now represented by 43,720 multi-omics features.


## Autoencoder Architecture

The concatenated multi-omics representation contains 43,720 features for each cell line.

The Autoencoder learns a compressed representation by passing the input through an encoder:

**43,720 → 4,096 → 1,024 → 512**

The final 512-dimensional vector is the latent representation of the cell line.

The decoder then attempts to reconstruct the original 43,720-dimensional input:

**512 → 1,024 → 4,096 → 43,720**

Therefore, the Autoencoder is trained to learn a compact representation while preserving information from the original multi-omics data.

### Reference architecture

The reference GraphTCDR Autoencoder uses:

- Linear layers
- Batch Normalization
- ReLU activation in hidden layers
- A 512-dimensional latent representation
- Sigmoid activation at the decoder output
- Mean Squared Error (MSE) reconstruction loss
- Adam optimizer
- Learning rate = 5 × 10⁻⁵
- 400 training epochs

The input data verified above are already scaled to the range [0, 1], which is consistent with using a Sigmoid activation at the decoder output.

### Important distinction

This notebook aims to implement the Autoencoder based on the reference GraphTCDR architecture.

Any changes to the reference training procedure, such as batch size, train/validation strategy, or other hyperparameters, will be explicitly documented as project adaptations rather than presented as exact reproduction.

Define GraphTCDR-Style Autoencoder

In [8]:
# ============================================================
# Notebook 08 — Cell 5
# Define GraphTCDR-Style Autoencoder
# ============================================================

import torch.nn as nn


class AutoEncoder(nn.Module):

    def __init__(
        self,
        input_dim=43720,
        hidden_dim_1=4096,
        hidden_dim_2=1024,
        latent_dim=512
    ):
        super().__init__()

        # ----------------------------------------------------
        # Encoder
        # 43,720 → 4,096 → 1,024 → 512
        # ----------------------------------------------------

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),

            nn.Linear(hidden_dim_2, latent_dim),
            nn.BatchNorm1d(latent_dim)
        )

        # ----------------------------------------------------
        # Decoder
        # 512 → 1,024 → 4,096 → 43,720
        # ----------------------------------------------------

        self.decoder = nn.Sequential(

            nn.Linear(latent_dim, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),

            nn.Linear(hidden_dim_2, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),

            nn.Linear(hidden_dim_1, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):

        latent = self.encoder(x)
        reconstructed = self.decoder(latent)

        return reconstructed


 Autoencoder Architecture Smoke Test

In [9]:
# Create a small batch from the actual multi-omics data
x_test = X[:4]

# Put the model in evaluation mode
model = AutoEncoder()
model.eval()

# Forward pass
with torch.no_grad():

    latent = model.encoder(x_test)
    reconstructed = model(x_test)


print("=" * 70)
print("AUTOENCODER SMOKE TEST")
print("=" * 70)

print(f"Input shape:          {tuple(x_test.shape)}")
print(f"Latent shape:         {tuple(latent.shape)}")
print(f"Reconstructed shape:  {tuple(reconstructed.shape)}")

print(f"\nInput range:")
print(f"  Minimum: {x_test.min().item():.6f}")
print(f"  Maximum: {x_test.max().item():.6f}")

print(f"\nReconstructed range:")
print(f"  Minimum: {reconstructed.min().item():.6f}")
print(f"  Maximum: {reconstructed.max().item():.6f}")


# ------------------------------------------------------------
# Shape checks
# ------------------------------------------------------------

if x_test.shape != (4, 43720):
    raise ValueError(
        f"Unexpected input shape: {tuple(x_test.shape)}"
    )

if latent.shape != (4, 512):
    raise ValueError(
        f"Unexpected latent shape: {tuple(latent.shape)}"
    )

if reconstructed.shape != (4, 43720):
    raise ValueError(
        f"Unexpected reconstructed shape: {tuple(reconstructed.shape)}"
    )


# ------------------------------------------------------------
# Output-range check
# ------------------------------------------------------------

if reconstructed.min().item() < 0 or reconstructed.max().item() > 1:
    raise ValueError(
        "Reconstructed values are outside the expected [0, 1] range."
    )


print("\n" + "=" * 70)
print("SMOKE TEST PASSED")
print("=" * 70)

print(
    "The Autoencoder successfully maps "
    "43,720-dimensional inputs to 512-dimensional "
    "latent representations and reconstructs them."
)

AUTOENCODER SMOKE TEST
Input shape:          (4, 43720)
Latent shape:         (4, 512)
Reconstructed shape:  (4, 43720)

Input range:
  Minimum: 0.000000
  Maximum: 1.000000

Reconstructed range:
  Minimum: 0.486439
  Maximum: 0.513044

SMOKE TEST PASSED
The Autoencoder successfully maps 43,720-dimensional inputs to 512-dimensional latent representations and reconstructs them.


Reproducibility and Training Configuration

In [10]:
import random
import numpy as np
import torch






In [11]:
# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [12]:

 #device

 DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [13]:
# Autoencoder training configuration

INPUT_DIM = 43720
HIDDEN_DIM_1 = 4096
HIDDEN_DIM_2 = 1024
LATENT_DIM = 512

LEARNING_RATE = 5e-5
EPOCHS = 400
BATCH_SIZE = 64

In [14]:
# Display configuration
print("=" * 70)
print("AUTOENCODER TRAINING CONFIGURATION")
print("=" * 70)

print(f"Random seed:       {SEED}")
print(f"Device:            {DEVICE}")

print("\nArchitecture:")
print(f"Input dimension:   {INPUT_DIM}")
print(f"Hidden layer 1:    {HIDDEN_DIM_1}")
print(f"Hidden layer 2:    {HIDDEN_DIM_2}")
print(f"Latent dimension:  {LATENT_DIM}")

print("\nTraining:")
print(f"Learning rate:     {LEARNING_RATE}")
print(f"Epochs:            {EPOCHS}")
print(f"Batch size:        {BATCH_SIZE}")

print("\nConfiguration ready.")

AUTOENCODER TRAINING CONFIGURATION
Random seed:       42
Device:            cuda

Architecture:
Input dimension:   43720
Hidden layer 1:    4096
Hidden layer 2:    1024
Latent dimension:  512

Training:
Learning rate:     5e-05
Epochs:            400
Batch size:        64

Configuration ready.


Create Autoencoder Train / Holdout Split

In [15]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [16]:
#convert cell-line indices into train/holdout sets
cell_indices = np.arange(X.shape[0])

train_indices, holdout_indices = train_test_split(
    cell_indices,
    test_size=40,
    random_state=SEED,
    shuffle=True
)

In [17]:
# Create tensors
X_train_ae = X[train_indices]
X_holdout_ae = X[holdout_indices]

In [18]:
# Create PyTorch datasets
train_dataset = TensorDataset(X_train_ae)
holdout_dataset = TensorDataset(X_holdout_ae)

In [19]:
# Create DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
holdout_loader = DataLoader(
    holdout_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [20]:
# Display split information
print("=" * 70)
print("AUTOENCODER TRAIN / HOLDOUT SPLIT")
print("=" * 70)

print(f"Total cell lines:       {len(cell_indices)}")
print(f"Training cell lines:    {len(train_indices)}")
print(f"Holdout cell lines:     {len(holdout_indices)}")

print("\nTraining matrix shape:")
print(tuple(X_train_ae.shape))

print("\nHoldout matrix shape:")
print(tuple(X_holdout_ae.shape))

print("\nDataLoader batches:")
print(f"Training batches:       {len(train_loader)}")
print(f"Holdout batches:        {len(holdout_loader)}")

AUTOENCODER TRAIN / HOLDOUT SPLIT
Total cell lines:       402
Training cell lines:    362
Holdout cell lines:     40

Training matrix shape:
(362, 43720)

Holdout matrix shape:
(40, 43720)

DataLoader batches:
Training batches:       6
Holdout batches:        1


In [21]:
# Verify no overlap
overlap = set(train_indices).intersection(set(holdout_indices))

if len(overlap) != 0:
    raise ValueError(
        f"Train/holdout overlap detected: {overlap}"
    )

print("\nNo cell-line overlap detected.")
print("Train/holdout split verified successfully.")


No cell-line overlap detected.
Train/holdout split verified successfully.


Training Smoke Test

In [41]:
#########
import torch.optim as optim


# ------------------------------------------------------------
# Create a fresh model
# ------------------------------------------------------------

model = AutoEncoder(
    input_dim=INPUT_DIM,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2,
    latent_dim=LATENT_DIM
).to(DEVICE)


# ------------------------------------------------------------
# Loss function and optimizer
# ------------------------------------------------------------

criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ------------------------------------------------------------
# Run a very short training test
# ------------------------------------------------------------

SMOKE_TEST_EPOCHS = 3

print("=" * 70)
print("AUTOENCODER TRAINING SMOKE TEST")
print("=" * 70)

print(f"Device: {DEVICE}")
print(f"Smoke-test epochs: {SMOKE_TEST_EPOCHS}")

for epoch in range(SMOKE_TEST_EPOCHS):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for (batch,) in train_loader:

        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        reconstructed = model(batch)

        loss = criterion(reconstructed, batch)

        loss.backward()

        optimizer.step()

        batch_size = batch.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples

    # --------------------------------------------------------
    # Holdout evaluation
    # --------------------------------------------------------

    model.eval()

    holdout_loss = 0.0
    holdout_samples = 0

    with torch.no_grad():

        for (batch,) in holdout_loader:

            batch = batch.to(DEVICE)

            reconstructed = model(batch)

            loss = criterion(reconstructed, batch)

            batch_size = batch.size(0)

            holdout_loss += loss.item() * batch_size
            holdout_samples += batch_size

    holdout_loss = holdout_loss / holdout_samples

    print(
        f"Epoch {epoch + 1}/{SMOKE_TEST_EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Holdout MSE: {holdout_loss:.6f}"
    )


print("\n" + "=" * 70)
print("TRAINING SMOKE TEST COMPLETED")
print("=" * 70)

AUTOENCODER TRAINING SMOKE TEST
Device: cuda
Smoke-test epochs: 3
Epoch 1/3 | Train MSE: 0.060644 | Holdout MSE: 0.058249
Epoch 2/3 | Train MSE: 0.040930 | Holdout MSE: 0.049802
Epoch 3/3 | Train MSE: 0.031506 | Holdout MSE: 0.039315

TRAINING SMOKE TEST COMPLETED


Full Autoencoder Training

In [22]:
import sys
import torch

print("=" * 70)
print("NOTEBOOK PYTHON / GPU CHECK")
print("=" * 70)

print("Python executable:")
print(sys.executable)

print("\nPyTorch version:")
print(torch.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

print("\nPyTorch CUDA version:")
print(torch.version.cuda)

print("\nGPU:")
print(
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "No GPU detected"
)

print("\nCurrent DEVICE variable:")
print(DEVICE)

NOTEBOOK PYTHON / GPU CHECK
Python executable:
d:\Pancreatic_GraphTCDR\.venv\Scripts\python.exe

PyTorch version:
2.11.0+cu128

CUDA available:
True

PyTorch CUDA version:
12.8

GPU:
NVIDIA GeForce RTX 3050 A Laptop GPU

Current DEVICE variable:
cuda


In [23]:
import sys
import subprocess

print("Notebook Python:")
print(sys.executable)

print("\nWhat a NEW Python process using the SAME executable sees:")

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.__file__)"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("ERROR:")
    print(result.stderr)

Notebook Python:
d:\Pancreatic_GraphTCDR\.venv\Scripts\python.exe

What a NEW Python process using the SAME executable sees:
2.11.0+cu128
True
d:\Pancreatic_GraphTCDR\.venv\lib\site-packages\torch\__init__.py



In [24]:
import torch
import sys

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: d:\Pancreatic_GraphTCDR\.venv\Scripts\python.exe
PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU


GPU training smoke test

In [25]:
import torch
import torch.nn as nn
import torch.optim as optim


In [26]:


import torch
import torch.nn as nn
import torch.optim as optim


# ------------------------------------------------------------
# Confirm GPU


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Stop here and check the GPU setup."
    )

DEVICE = torch.device("cuda")

print("=" * 70)
print("AUTOENCODER GPU TRAINING SMOKE TEST")
print("=" * 70)

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


# ------------------------------------------------------------
# Create a fresh Autoencoder


model = AutoEncoder(
    input_dim=INPUT_DIM,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2,
    latent_dim=LATENT_DIM
).to(DEVICE)


# ------------------------------------------------------------
# Loss and optimizer


criterion = nn.MSELoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ------------------------------------------------------------
# GPU training smoke test


SMOKE_TEST_EPOCHS = 3

for epoch in range(SMOKE_TEST_EPOCHS):

    model.train()

    running_loss = 0.0
    total_samples = 0

    for (batch,) in train_loader:

        batch = batch.to(DEVICE)

        optimizer.zero_grad()

        reconstructed = model(batch)

        loss = criterion(reconstructed, batch)

        loss.backward()

        optimizer.step()

        batch_size = batch.size(0)

        running_loss += loss.item() * batch_size
        total_samples += batch_size

    train_loss = running_loss / total_samples


    # --------------------------------------------------------
    # Holdout evaluation
   

    model.eval()

    holdout_loss = 0.0
    holdout_samples = 0

    with torch.no_grad():

        for (batch,) in holdout_loader:

            batch = batch.to(DEVICE)

            reconstructed = model(batch)

            loss = criterion(reconstructed, batch)

            batch_size = batch.size(0)

            holdout_loss += loss.item() * batch_size
            holdout_samples += batch_size

    holdout_loss = holdout_loss / holdout_samples


    # --------------------------------------------------------
    # GPU memory information

    gpu_memory = torch.cuda.memory_allocated() / (1024 ** 3)
    gpu_reserved = torch.cuda.memory_reserved() / (1024 ** 3)

    print(
        f"Epoch {epoch + 1}/{SMOKE_TEST_EPOCHS} | "
        f"Train MSE: {train_loss:.6f} | "
        f"Holdout MSE: {holdout_loss:.6f} | "
        f"GPU allocated: {gpu_memory:.2f} GB | "
        f"GPU reserved: {gpu_reserved:.2f} GB"
    )


print("\n" + "=" * 70)
print("GPU TRAINING SMOKE TEST COMPLETED")
print("=" * 70)

AUTOENCODER GPU TRAINING SMOKE TEST
Device: cuda
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU
Epoch 1/3 | Train MSE: 0.060644 | Holdout MSE: 0.058249 | GPU allocated: 6.89 GB | GPU reserved: 8.96 GB
Epoch 2/3 | Train MSE: 0.040930 | Holdout MSE: 0.049802 | GPU allocated: 6.89 GB | GPU reserved: 8.96 GB
Epoch 3/3 | Train MSE: 0.031506 | Holdout MSE: 0.039315 | GPU allocated: 6.89 GB | GPU reserved: 8.96 GB

GPU TRAINING SMOKE TEST COMPLETED


In [36]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

print("=" * 70)
print("FULL AUTOENCODER TRAINING")
print("=" * 70)

model = AutoEncoder(
    input_dim=INPUT_DIM,
    hidden_dim_1=HIDDEN_DIM_1,
    hidden_dim_2=HIDDEN_DIM_2,
    latent_dim=LATENT_DIM
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

train_losses = []
holdout_losses = []

best_holdout_loss = float("inf")
best_state_dict = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):

    # -------------------------
    # Training
    # -------------------------
    model.train()
    train_loss = 0.0

    for (batch_x,) in train_loader:
        batch_x = batch_x.to(DEVICE)

        optimizer.zero_grad()

        reconstructed = model(batch_x)
        loss = criterion(reconstructed, batch_x)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)

    train_loss /= len(train_loader.dataset)

    # -------------------------
    # Holdout evaluation
    # -------------------------
    model.eval()
    holdout_loss = 0.0

    with torch.no_grad():
        for (batch_x,) in holdout_loader:
            batch_x = batch_x.to(DEVICE)

            reconstructed = model(batch_x)
            loss = criterion(reconstructed, batch_x)

            holdout_loss += loss.item() * batch_x.size(0)

    holdout_loss /= len(holdout_loader.dataset)

    train_losses.append(train_loss)
    holdout_losses.append(holdout_loss)

    # -------------------------
    # Save best model
    # -------------------------
    if holdout_loss < best_holdout_loss:
        best_holdout_loss = holdout_loss
        best_epoch = epoch
        best_state_dict = copy.deepcopy(model.state_dict())

    # -------------------------
    # Progress
    # -------------------------
    if epoch == 1 or epoch % 10 == 0:
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3

        print(
            f"Epoch {epoch:3d}/{EPOCHS} | "
            f"Train MSE: {train_loss:.6f} | "
            f"Holdout MSE: {holdout_loss:.6f} | "
            f"Best: {best_holdout_loss:.6f} "
            f"(epoch {best_epoch}) | "
            f"GPU: {allocated:.2f} GB"
        )

# Restore best model
model.load_state_dict(best_state_dict)
model.eval()

print("\n" + "=" * 70)
print("FULL TRAINING COMPLETED")
print("=" * 70)
print(f"Best epoch: {best_epoch}")
print(f"Best holdout MSE: {best_holdout_loss:.6f}")

FULL AUTOENCODER TRAINING
Epoch   1/400 | Train MSE: 0.060681 | Holdout MSE: 0.058370 | Best: 0.058370 (epoch 1) | GPU: 6.91 GB
Epoch  10/400 | Train MSE: 0.019224 | Holdout MSE: 0.020158 | Best: 0.020158 (epoch 10) | GPU: 6.91 GB
Epoch  20/400 | Train MSE: 0.015699 | Holdout MSE: 0.018056 | Best: 0.018056 (epoch 20) | GPU: 6.91 GB
Epoch  30/400 | Train MSE: 0.013463 | Holdout MSE: 0.017081 | Best: 0.017081 (epoch 30) | GPU: 6.91 GB
Epoch  40/400 | Train MSE: 0.011546 | Holdout MSE: 0.016409 | Best: 0.016409 (epoch 40) | GPU: 6.91 GB
Epoch  50/400 | Train MSE: 0.010155 | Holdout MSE: 0.016022 | Best: 0.016022 (epoch 50) | GPU: 6.91 GB
Epoch  60/400 | Train MSE: 0.008924 | Holdout MSE: 0.015761 | Best: 0.015746 (epoch 59) | GPU: 6.91 GB
Epoch  70/400 | Train MSE: 0.007867 | Holdout MSE: 0.015580 | Best: 0.015579 (epoch 69) | GPU: 6.91 GB
Epoch  80/400 | Train MSE: 0.006958 | Holdout MSE: 0.015443 | Best: 0.015428 (epoch 78) | GPU: 6.91 GB
Epoch  90/400 | Train MSE: 0.006153 | Holdout MS

In [27]:
# Save the newly trained best Autoencoder checkpoint

import os
import torch

MODEL_DIR = os.path.join(
    PROJECT,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "autoencoder_best.pt"
)

torch.save(
    {
        "model_state_dict": best_state_dict,
        "input_dim": INPUT_DIM,
        "hidden_dim_1": HIDDEN_DIM_1,
        "hidden_dim_2": HIDDEN_DIM_2,
        "latent_dim": LATENT_DIM,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "best_epoch": best_epoch,
        "best_holdout_mse": best_holdout_loss,
        "seed": SEED,
    },
    MODEL_PATH
)

print("=" * 70)
print("UPDATED AUTOENCODER CHECKPOINT")
print("=" * 70)

print("Best epoch:", best_epoch)
print("Best holdout MSE:", best_holdout_loss)
print("Saved to:", MODEL_PATH)
print("File exists:", os.path.exists(MODEL_PATH))

print("\n✓ NEW BEST AUTOENCODER SAVED")

NameError: name 'best_state_dict' is not defined

In [28]:
# Select the available computation device

import torch

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("COMPUTATION DEVICE")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\n✓ DEVICE CONFIGURED")

COMPUTATION DEVICE
PyTorch version: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 3050 A Laptop GPU

✓ DEVICE CONFIGURED


## Restore the Best Autoencoder Checkpoint

The Autoencoder has already been trained for 400 epochs. We do not retrain it here.

Instead, we load the checkpoint corresponding to the lowest holdout reconstruction
error observed during training.

The selected checkpoint is:

- Best epoch: 388
- Best holdout MSE: 0.014964

This restored model will be used only to generate the final 512-dimensional
representations of the 28 pancreatic cancer cell lines.

In [29]:
# Restore the best trained Autoencoder

import torch

checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE
)

MODEL = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    hidden_dim_1=checkpoint["hidden_dim_1"],
    hidden_dim_2=checkpoint["hidden_dim_2"],
    latent_dim=checkpoint["latent_dim"]
).to(DEVICE)

MODEL.load_state_dict(
    checkpoint["model_state_dict"]
)

MODEL.eval()

print("=" * 70)
print("BEST AUTOENCODER RESTORED")
print("=" * 70)

print("Best epoch:", checkpoint["best_epoch"])
print("Best holdout MSE:", checkpoint["best_holdout_mse"])
print("Device:", DEVICE)

print("\n✓ EPOCH-388 BEST MODEL RESTORED")

BEST AUTOENCODER RESTORED
Best epoch: 388
Best holdout MSE: 0.01496440451592207
Device: cuda

✓ EPOCH-388 BEST MODEL RESTORED


## Generate Final 512-Dimensional Pancreatic Cell-Line Embeddings

The best Autoencoder checkpoint (epoch 388) is now restored.

We next extract the encoder representation for the 28 pancreatic cancer
cell lines identified in the GraphTCDR-compatible 402-cell dataset.

Only the encoder is used:

    43,720-dimensional multi-omics input
                    ↓
              Autoencoder
                    ↓
          512-dimensional embedding

These 512-dimensional embeddings will serve as the cell-line node features
for the downstream heterogeneous graph model.

The embeddings are generated from the best checkpoint rather than from the
final training epoch, because epoch 388 achieved the lowest holdout MSE.

In [30]:
# Generate final 512-dimensional embeddings for the 28 pancreatic cell lines

import os
import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# 1. Load the pancreatic cell-line mapping
# ------------------------------------------------------------
mapping_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_mapping.csv"
)

mapping = pd.read_csv(mapping_path)

print("=" * 70)
print("PANCREATIC CELL-LINE MAPPING")
print("=" * 70)
print("Number of pancreatic cell lines:", len(mapping))
print("Columns:", mapping.columns.tolist())


# ------------------------------------------------------------
# 2. Obtain the original 402-cell indices
# ------------------------------------------------------------
pancreatic_indices = mapping["index"].astype(int).to_numpy()

print("\nOriginal 402-cell indices:")
print(pancreatic_indices)


# ------------------------------------------------------------
# 3. Move the corresponding multi-omics rows to the GPU
# ------------------------------------------------------------
X_pancreatic = X[pancreatic_indices].to(DEVICE)

print("\nPancreatic input shape:", tuple(X_pancreatic.shape))


# ------------------------------------------------------------
# 4. Extract the 512-dimensional encoder representation
# ------------------------------------------------------------
with torch.no_grad():
    pancreatic_embeddings = MODEL.encoder(X_pancreatic)

pancreatic_embeddings = pancreatic_embeddings.cpu().numpy()


# ------------------------------------------------------------
# 5. Verify the generated embeddings
# ------------------------------------------------------------
print("\nGenerated embedding shape:",
      pancreatic_embeddings.shape)

print("Minimum value:", pancreatic_embeddings.min())
print("Maximum value:", pancreatic_embeddings.max())

print("NaN values:",
      np.isnan(pancreatic_embeddings).sum())

print("Inf values:",
      np.isinf(pancreatic_embeddings).sum())


# ------------------------------------------------------------
# 6. Save the embeddings
# ------------------------------------------------------------
embedding_columns = [
    f"AE_{i}"
    for i in range(1, pancreatic_embeddings.shape[1] + 1)
]

embedding_df = pd.DataFrame(
    pancreatic_embeddings,
    columns=embedding_columns
)

embedding_df.insert(
    0,
    "cellLineName",
    mapping["cellLineName"].values
)

embedding_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_ae_embeddings_512.csv"
)

embedding_df.to_csv(
    embedding_path,
    index=False
)

print("\nSaved to:")
print(embedding_path)

print("\n✓ FINAL PANCREATIC EMBEDDINGS GENERATED")
print("✓ 28 cell lines × 512 latent features")
print("✓ Generated from best epoch-388 checkpoint")

PANCREATIC CELL-LINE MAPPING
Number of pancreatic cell lines: 28
Columns: ['index', 'cellLineName']

Original 402-cell indices:
[  0  39  47  68  79  84  85  91 103 123 135 160 165 166 179 234 237 266
 291 294 313 319 321 333 362 384 386 400]

Pancreatic input shape: (28, 43720)

Generated embedding shape: (28, 512)
Minimum value: -2.6833937
Maximum value: 2.8198113
NaN values: 0
Inf values: 0

Saved to:
D:\Pancreatic_GraphTCDR\data\processed\pancreatic28_ae_embeddings_512.csv

✓ FINAL PANCREATIC EMBEDDINGS GENERATED
✓ 28 cell lines × 512 latent features
✓ Generated from best epoch-388 checkpoint


## Reload Final Cell-Node Features

The pancreatic cell-line embeddings have been regenerated using the best
Autoencoder checkpoint (epoch 388).

We now reload these embeddings as the cell-node features that will be used
by the downstream heterogeneous graph model.

This ensures that the graph model does not accidentally use embeddings from
an earlier Autoencoder training run.

In [31]:
# Reload the final epoch-388 cell embeddings

import os
import numpy as np
import pandas as pd

embedding_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_ae_embeddings_512.csv"
)

ae_df = pd.read_csv(embedding_path)

cell_features = ae_df.drop(
    columns=["cellLineName"]
).to_numpy(dtype=np.float32)

print("=" * 70)
print("FINAL CELL NODE FEATURES")
print("=" * 70)

print("Embedding table shape:", ae_df.shape)
print("Cell feature matrix shape:", cell_features.shape)

print("NaN values:", np.isnan(cell_features).sum())
print("Inf values:", np.isinf(cell_features).sum())

print("\nFirst 5 cell lines:")
print(ae_df["cellLineName"].head().tolist())

print("\n✓ FINAL EPOCH-388 EMBEDDINGS LOADED")
print("✓ Cell-node feature matrix is 28 × 512")

FINAL CELL NODE FEATURES
Embedding table shape: (28, 513)
Cell feature matrix shape: (28, 512)
NaN values: 0
Inf values: 0

First 5 cell lines:
['PATU8988S_PANCREAS', 'PANC0327_PANCREAS', 'PANC1_PANCREAS', 'PSN1_PANCREAS', 'SUIT2_PANCREAS']

✓ FINAL EPOCH-388 EMBEDDINGS LOADED
✓ Cell-node feature matrix is 28 × 512


## Verify the Frozen Cold-Drug Split

The same fixed drug-level split is used for the entire modelling pipeline.

The purpose of this split is to evaluate whether the model can predict
responses for drugs that were completely unseen during training.

Expected split:

- 1,228 total drugs
- 982 training drugs
- 246 test drugs
- 14,235 training cell–drug pairs
- 3,802 test cell–drug pairs
- 0 overlapping drugs between train and test
- 28 pancreatic cell lines represented in both sets

This verification is performed before constructing the GraphTCDR model to
ensure that the downstream experiment preserves the intended cold-drug
evaluation protocol.

In [33]:
# Verify the frozen cold-drug split

import os
import pandas as pd

TRAIN_SPLIT_PATH = os.path.join(
    PROJECT,
    "data",
    "processed",
    "splits",
    "cold_drug_train.csv"
)

TEST_SPLIT_PATH = os.path.join(
    PROJECT,
    "data",
    "processed",
    "splits",
    "cold_drug_test.csv"
)

train_df = pd.read_csv(TRAIN_SPLIT_PATH)
test_df = pd.read_csv(TEST_SPLIT_PATH)

train_drugs = set(train_df["drug_index"].unique())
test_drugs = set(test_df["drug_index"].unique())

train_cells = set(train_df["omics_index"].unique())
test_cells = set(test_df["omics_index"].unique())

drug_overlap = train_drugs.intersection(test_drugs)

print("=" * 70)
print("FROZEN COLD-DRUG SPLIT VERIFICATION")
print("=" * 70)

print("\nTraining pairs:", len(train_df))
print("Testing pairs:", len(test_df))

print("\nTraining drugs:", len(train_drugs))
print("Testing drugs:", len(test_drugs))
print("Total unique drugs:", len(train_drugs | test_drugs))

print("\nTraining cells:", len(train_cells))
print("Testing cells:", len(test_cells))

print("\nDrug overlap:", len(drug_overlap))

print("\nExpected:")
print("Train drugs = 982")
print("Test drugs  = 246")
print("Train pairs = 14,235")
print("Test pairs  = 3,802")
print("Drug overlap = 0")
print("Cells in train/test = 28")

# Explicit assertions
assert len(train_df) == 14235
assert len(test_df) == 3802
assert len(train_drugs) == 982
assert len(test_drugs) == 246
assert len(train_drugs | test_drugs) == 1228
assert len(drug_overlap) == 0
assert len(train_cells) == 28
assert len(test_cells) == 28

print("\n" + "=" * 70)
print("✓ ALL COLD-DRUG SPLIT CHECKS PASSED")
print("=" * 70)

FROZEN COLD-DRUG SPLIT VERIFICATION

Training pairs: 14235
Testing pairs: 3802

Training drugs: 982
Testing drugs: 246
Total unique drugs: 1228

Training cells: 28
Testing cells: 28

Drug overlap: 0

Expected:
Train drugs = 982
Test drugs  = 246
Train pairs = 14,235
Test pairs  = 3,802
Drug overlap = 0
Cells in train/test = 28

✓ ALL COLD-DRUG SPLIT CHECKS PASSED


## Verify the 402-Cell to 28-Cell Node Mapping

The GraphTCDR multi-omics dataset uses the original 402-cell-line indexing.

Our pancreatic graph contains only the 28 cell lines that were successfully
matched between PRISM and the GraphTCDR features.

Therefore, the original 402-cell indices cannot be used directly as graph
node indices.

For example:

    Original GraphTCDR index 39
                    ↓
          Local pancreatic node index 1

The mapping file explicitly provides:

    index → cellLineName

We therefore construct and verify an explicit mapping:

    original 402-cell index → local 0–27 graph node index

This mapping will be used when constructing the response-derived cell–drug
edges.

No index conversion will be inferred from ordering alone.

In [34]:
# Verify and construct the 402-cell -> 28-cell graph-node mapping

import os
import pandas as pd
import numpy as np

MAPPING_PATH = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_mapping.csv"
)

mapping_check = pd.read_csv(MAPPING_PATH)

print("=" * 70)
print("402-CELL → 28-CELL NODE MAPPING")
print("=" * 70)

print("Mapping shape:", mapping_check.shape)
print("\nMapping columns:", mapping_check.columns.tolist())

# ------------------------------------------------------------
# Construct explicit mapping
# ------------------------------------------------------------

original_to_local = {
    int(row["index"]): local_idx
    for local_idx, (_, row) in enumerate(mapping_check.iterrows())
}

local_to_original = {
    local_idx: int(row["index"])
    for local_idx, (_, row) in enumerate(mapping_check.iterrows())
}

# ------------------------------------------------------------
# Verify the mapping
# ------------------------------------------------------------

print("\nNumber of original indices:", len(original_to_local))
print("Number of local nodes:", len(local_to_original))

print("\nFirst 10 mappings:")
for original_idx, local_idx in list(original_to_local.items())[:10]:
    cell_name = mapping_check.iloc[local_idx]["cellLineName"]
    print(
        f"Original index {original_idx:>3} "
        f"→ Local node {local_idx:>2} "
        f"→ {cell_name}"
    )

# ------------------------------------------------------------
# Consistency checks
# ------------------------------------------------------------

assert len(mapping_check) == 28
assert len(original_to_local) == 28
assert len(local_to_original) == 28

assert len(set(original_to_local.values())) == 28
assert len(set(local_to_original.values())) == 28

# Every local node must have a corresponding cell name
assert mapping_check["cellLineName"].notna().all()

print("\n" + "=" * 70)
print("✓ 402 → 28 INDEX MAPPING VERIFIED")
print("=" * 70)

print("\nOriginal index range:")
print(
    min(original_to_local.keys()),
    "to",
    max(original_to_local.keys())
)

print("Local graph-node range:")
print(
    min(original_to_local.values()),
    "to",
    max(original_to_local.values())
)

402-CELL → 28-CELL NODE MAPPING
Mapping shape: (28, 2)

Mapping columns: ['index', 'cellLineName']

Number of original indices: 28
Number of local nodes: 28

First 10 mappings:
Original index   0 → Local node  0 → PATU8988S_PANCREAS
Original index  39 → Local node  1 → PANC0327_PANCREAS
Original index  47 → Local node  2 → PANC1_PANCREAS
Original index  68 → Local node  3 → PSN1_PANCREAS
Original index  79 → Local node  4 → SUIT2_PANCREAS
Original index  84 → Local node  5 → SNU213_PANCREAS
Original index  85 → Local node  6 → PANC0813_PANCREAS
Original index  91 → Local node  7 → DANG_PANCREAS
Original index 103 → Local node  8 → HUPT3_PANCREAS
Original index 123 → Local node  9 → CAPAN2_PANCREAS

✓ 402 → 28 INDEX MAPPING VERIFIED

Original index range:
0 to 400
Local graph-node range:
0 to 27


## Verify the Training-Only Cell–Drug Response Graph

GraphTCDR uses a heterogeneous graph containing cell–cell, drug–drug and
cell–drug relationships.

For the cold-drug experiment, the response-derived cell–drug edges must be
constructed using training responses only.

This is a deliberate leakage-control adaptation:

    Training response data
            ↓
    Response-derived edges
            ↓
    Graph used for training

The held-out test drugs must not contribute response-derived edges.

The response graph was constructed using the lower 50% of training response
values for each training drug, following the response-edge construction
principle identified in the GraphTCDR repository.

Because the original multi-omics dataset uses 402-cell indices, the response
edges must also be converted to local pancreatic graph-node indices (0–27).

In [35]:
# Verify the training-only response graph after 402 -> 28 index conversion

import os
import numpy as np
import pandas as pd

RESPONSE_EDGE_PATH = os.path.join(
    PROJECT,
    "data",
    "processed",
    "train_response_cell_drug_edges_lower50.csv"
)

response_edges_raw = pd.read_csv(RESPONSE_EDGE_PATH)

print("=" * 70)
print("TRAINING-ONLY RESPONSE GRAPH VERIFICATION")
print("=" * 70)

print("Saved edge table shape:", response_edges_raw.shape)
print("Columns:", response_edges_raw.columns.tolist())

# ------------------------------------------------------------
# Convert original 402-cell indices to local 0–27 indices
# ------------------------------------------------------------

response_edges_local = response_edges_raw.copy()

response_edges_local["cell_local"] = (
    response_edges_local["omics_index"]
    .map(original_to_local)
)

# ------------------------------------------------------------
# Check that every original cell index was mapped
# ------------------------------------------------------------

missing_mappings = response_edges_local[
    response_edges_local["cell_local"].isna()
]

print("\nEdges with missing cell mapping:",
      len(missing_mappings))

assert len(missing_mappings) == 0

# ------------------------------------------------------------
# Convert to integer local indices
# ------------------------------------------------------------

response_edges_local["cell_local"] = (
    response_edges_local["cell_local"]
    .astype(int)
)

# ------------------------------------------------------------
# Basic graph statistics
# ------------------------------------------------------------

unique_cells = set(
    response_edges_local["cell_local"].unique()
)

unique_drugs = set(
    response_edges_local["drug_index"].unique()
)

print("\nNumber of response edges:",
      len(response_edges_local))

print("Unique local cell nodes:",
      len(unique_cells))

print("Local cell-node range:",
      min(unique_cells),
      "to",
      max(unique_cells))

print("Unique drugs in response graph:",
      len(unique_drugs))

print("Drug-index range:",
      min(unique_drugs),
      "to",
      max(unique_drugs))

# ------------------------------------------------------------
# Check local cell-node validity
# ------------------------------------------------------------

assert min(unique_cells) >= 0
assert max(unique_cells) < 28
assert len(unique_cells) == 28

# ------------------------------------------------------------
# Check that every response-graph drug is a training drug
# ------------------------------------------------------------

response_drug_overlap_with_test = (
    unique_drugs.intersection(test_drugs)
)

response_drug_overlap_with_train = (
    unique_drugs.intersection(train_drugs)
)

print("\nResponse-graph drugs belonging to training set:",
      len(response_drug_overlap_with_train))

print("Response-graph drugs belonging to test set:",
      len(response_drug_overlap_with_test))

assert len(response_drug_overlap_with_test) == 0

# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert len(response_edges_local) == 7370
assert len(unique_cells) == 28
assert len(response_drug_overlap_with_test) == 0

print("\n" + "=" * 70)
print("✓ RESPONSE GRAPH AUDIT PASSED")
print("=" * 70)

print("\nVerified:")
print("✓ 7,370 response-derived edges")
print("✓ All 28 pancreatic cells represented")
print("✓ Local cell indices are 0–27")
print("✓ Response graph contains training drugs only")
print("✓ Zero test-drug response edges")
print("✓ No missing 402 → 28 mappings")

TRAINING-ONLY RESPONSE GRAPH VERIFICATION
Saved edge table shape: (7370, 2)
Columns: ['omics_index', 'drug_index']

Edges with missing cell mapping: 0

Number of response edges: 7370
Unique local cell nodes: 28
Local cell-node range: 0 to 27
Unique drugs in response graph: 982
Drug-index range: 0 to 1226

Response-graph drugs belonging to training set: 982
Response-graph drugs belonging to test set: 0

✓ RESPONSE GRAPH AUDIT PASSED

Verified:
✓ 7,370 response-derived edges
✓ All 28 pancreatic cells represented
✓ Local cell indices are 0–27
✓ Response graph contains training drugs only
✓ Zero test-drug response edges
✓ No missing 402 → 28 mappings


In [1]:
# Load the previously trained Autoencoder checkpoint

import os
import torch

MODEL_PATH = r"D:\Pancreatic_GraphTCDR\data\processed\autoencoder_best.pt"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Autoencoder checkpoint not found:\n{MODEL_PATH}"
    )

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu"
)

print("=" * 70)
print("AUTOENCODER CHECKPOINT LOADED")
print("=" * 70)

print("Best epoch:", checkpoint["best_epoch"])
print("Best holdout MSE:", checkpoint["best_holdout_mse"])
print("Input dimension:", checkpoint["input_dim"])
print("Latent dimension:", checkpoint["latent_dim"])

print("\n✓ CHECKPOINT FOUND AND LOADED")

AUTOENCODER CHECKPOINT LOADED
Best epoch: 388
Best holdout MSE: 0.01496440451592207
Input dimension: 43720
Latent dimension: 512

✓ CHECKPOINT FOUND AND LOADED


In [31]:
# Recreate the Autoencoder architecture and restore the saved weights

import torch
import torch.nn as nn


class AutoEncoder(nn.Module):
    def __init__(
        self,
        input_dim=43720,
        hidden_dim_1=4096,
        hidden_dim_2=1024,
        latent_dim=512
    ):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),

            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),

            nn.Linear(hidden_dim_2, latent_dim),
            nn.BatchNorm1d(latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim_2),
            nn.BatchNorm1d(hidden_dim_2),
            nn.ReLU(),

            nn.Linear(hidden_dim_2, hidden_dim_1),
            nn.BatchNorm1d(hidden_dim_1),
            nn.ReLU(),

            nn.Linear(hidden_dim_1, input_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed


# Recreate model using the saved architecture
model = AutoEncoder(
    input_dim=checkpoint["input_dim"],
    hidden_dim_1=checkpoint["hidden_dim_1"],
    hidden_dim_2=checkpoint["hidden_dim_2"],
    latent_dim=checkpoint["latent_dim"]
)

# Restore trained weights
model.load_state_dict(checkpoint["model_state_dict"])

# Evaluation mode
model.eval()

print("=" * 70)
print("AUTOENCODER RESTORED")
print("=" * 70)

print("Input dimension:", checkpoint["input_dim"])
print("Latent dimension:", checkpoint["latent_dim"])
print("Best epoch:", checkpoint["best_epoch"])
print("Best holdout MSE:", checkpoint["best_holdout_mse"])

print("\n✓ TRAINED AUTOENCODER RESTORED SUCCESSFULLY")

AUTOENCODER RESTORED
Input dimension: 43720
Latent dimension: 512
Best epoch: 385
Best holdout MSE: 0.015101735480129719

✓ TRAINED AUTOENCODER RESTORED SUCCESSFULLY


extract the 512-dimensional representation

In [32]:
# Extract 512-dimensional autoencoder embeddings
# for the 28 pancreatic cancer cell lines

import pandas as pd
import numpy as np
import torch

MODEL = model
MODEL.eval()

# Load pancreatic cell-line mapping
mapping_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_mapping.csv"
)

mapping = pd.read_csv(mapping_path)

print("Mapping shape:", mapping.shape)
print(mapping.head())

# GraphTCDR feature files contain all 402 cell lines.
# 'index' tells us which rows correspond to the 28 pancreatic lines.
pancreatic_indices = mapping["index"].astype(int).to_numpy()

# Select the corresponding rows from the original 402-cell matrix
X_pancreatic = X[pancreatic_indices].to(DEVICE)

# Extract encoder representation
with torch.no_grad():
    pancreatic_embeddings = MODEL.encoder(X_pancreatic)

pancreatic_embeddings = pancreatic_embeddings.cpu().numpy()

print("\nEmbedding shape:", pancreatic_embeddings.shape)
print("Expected shape: (28, 512)")
print("NaN:", np.isnan(pancreatic_embeddings).sum())
print("Inf:", np.isinf(pancreatic_embeddings).sum())

if pancreatic_embeddings.shape != (28, 512):
    raise ValueError("Unexpected embedding shape.")

if not np.isfinite(pancreatic_embeddings).all():
    raise ValueError("Embedding contains NaN or Inf.")

print("\n512-dimensional pancreatic embeddings extracted successfully.")

Mapping shape: (28, 2)
   index        cellLineName
0      0  PATU8988S_PANCREAS
1     39   PANC0327_PANCREAS
2     47      PANC1_PANCREAS
3     68       PSN1_PANCREAS
4     79      SUIT2_PANCREAS


RuntimeError: Expected all tensors to be on the same device, but got mat1 is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA_addmm)

save the embeddings

In [33]:
# Save pancreatic 512-dimensional autoencoder embeddings

embedding_df = pd.DataFrame(
    pancreatic_embeddings,
    columns=[f"AE_{i+1}" for i in range(512)]
)

embedding_df.insert(
    0,
    "cellLineName",
    mapping["cellLineName"].values
)

embedding_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_ae_embeddings_512.csv"
)

embedding_df.to_csv(embedding_path, index=False)

print("Saved:", embedding_path)
print("Shape:", embedding_df.shape)
print("\nFirst 5 rows:")
display(embedding_df.head())

NameError: name 'pancreatic_embeddings' is not defined

In [34]:
# Verify AE embedding ↔ pancreatic cell-line mapping ↔ response data

import pandas as pd
import os

embedding_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_ae_embeddings_512.csv"
)

mapping_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_mapping.csv"
)

response_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "integrated_modeling_table.csv"
)

ae = pd.read_csv(embedding_path)
mapping_check = pd.read_csv(mapping_path)
response = pd.read_csv(response_path)

ae_cells = set(ae["cellLineName"])
mapping_cells = set(mapping_check["cellLineName"])
response_cells = set(response["ccle_name"])

print("AE embedding cell lines:", len(ae_cells))
print("Mapping cell lines:", len(mapping_cells))
print("Response cell lines:", len(response_cells))

print("\nAE ↔ mapping:")
print("Missing from AE:", mapping_cells - ae_cells)
print("Extra in AE:", ae_cells - mapping_cells)

print("\nAE ↔ response:")
print("Missing from response:", ae_cells - response_cells)
print("Extra in response:", response_cells - ae_cells)

if (
    ae_cells == mapping_cells
    and ae_cells == response_cells
    and ae.shape[1] == 513
):
    print("\n✓ CELL-LINE MAPPING VERIFIED")
else:
    raise ValueError("Cell-line mappings do not match.")

print("\nResponse table shape:", response.shape)
print("AE embedding shape:", ae.shape)

AE embedding cell lines: 28
Mapping cell lines: 28
Response cell lines: 28

AE ↔ mapping:
Missing from AE: set()
Extra in AE: set()

AE ↔ response:
Missing from response: set()
Extra in response: set()

✓ CELL-LINE MAPPING VERIFIED

Response table shape: (18037, 7)
AE embedding shape: (28, 513)


prepare the graph inputs
Before constructing any graph, we need to establish the three separate components:
1. Cell–cell graph → based on methylation similarity
2. Drug–drug graph → based on molecular fingerprint similarity
3. Cell–drug response graph → training data only to prevent cold-drug leakage

In [ ]:
# Prepare the 28 pancreatic cell-line feature matrix
# from the trained autoencoder representation

import pandas as pd
import numpy as np

ae_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_ae_embeddings_512.csv"
)

ae_df = pd.read_csv(ae_path)

cell_names = ae_df["cellLineName"].values
cell_features = ae_df.drop(columns=["cellLineName"]).to_numpy(dtype=np.float32)

print("=" * 70)
print("CELL FEATURE MATRIX")
print("=" * 70)

print("Cell names:", len(cell_names))
print("Feature matrix shape:", cell_features.shape)
print("Expected:", (28, 512))

print("NaN:", np.isnan(cell_features).sum())
print("Inf:", np.isinf(cell_features).sum())

print("\nFirst 5 cell lines:")
print(cell_names[:5])

if cell_features.shape != (28, 512):
    raise ValueError("Unexpected cell feature shape.")

if not np.isfinite(cell_features).all():
    raise ValueError("Cell features contain NaN or Inf.")

print("\n✓ CELL FEATURE MATRIX VERIFIED")

CELL FEATURE MATRIX
Cell names: 28
Feature matrix shape: (28, 512)
Expected: (28, 512)
NaN: 0
Inf: 0

First 5 cell lines:
['PATU8988S_PANCREAS' 'PANC0327_PANCREAS' 'PANC1_PANCREAS' 'PSN1_PANCREAS'
 'SUIT2_PANCREAS']

✓ CELL FEATURE MATRIX VERIFIED


In [ ]:
# Load methylation features for constructing the cell-cell graph

import torch
import numpy as np
import os

methylation_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_methylation.pt"
)

if not os.path.exists(methylation_path):
    raise FileNotFoundError(methylation_path)

methylation = torch.load(
    methylation_path,
    map_location="cpu"
)

methylation = torch.as_tensor(methylation).float().numpy()

print("=" * 70)
print("PANCREATIC METHYLATION FEATURES")
print("=" * 70)

print("Shape:", methylation.shape)
print("Expected:", (28, 20587))
print("Min:", methylation.min())
print("Max:", methylation.max())
print("NaN:", np.isnan(methylation).sum())
print("Inf:", np.isinf(methylation).sum())

if methylation.shape != (28, 20587):
    raise ValueError("Unexpected methylation shape.")

if not np.isfinite(methylation).all():
    raise ValueError("Methylation matrix contains NaN or Inf.")

print("\n✓ METHYLATION FEATURES VERIFIED")

PANCREATIC METHYLATION FEATURES
Shape: (28, 20587)
Expected: (28, 20587)
Min: 0.0
Max: 1.0
NaN: 0
Inf: 0

✓ METHYLATION FEATURES VERIFIED


cell–cell similarity matrix using the GraphTCDR rule: Pearson correlation of methylation profiles, with the paper's absolute correlation threshold of 0.75

In [ ]:
# Construct the cell-cell similarity graph
# GraphTCDR rule: |Pearson correlation| >= 0.75

from scipy.stats import pearsonr
import numpy as np
import pandas as pd

n_cells = methylation.shape[0]

cell_similarity = np.zeros((n_cells, n_cells), dtype=np.float32)

for i in range(n_cells):
    for j in range(i, n_cells):

        r, _ = pearsonr(
            methylation[i],
            methylation[j]
        )

        cell_similarity[i, j] = r
        cell_similarity[j, i] = r

# GraphTCDR threshold
CELL_THRESHOLD = 0.75

cell_adjacency = (
    np.abs(cell_similarity) >= CELL_THRESHOLD
).astype(np.float32)

# Remove self-loops for the graph edge construction
np.fill_diagonal(cell_adjacency, 0)

print("=" * 70)
print("CELL-CELL SIMILARITY GRAPH")
print("=" * 70)

print("Similarity matrix shape:", cell_similarity.shape)
print("Threshold:", CELL_THRESHOLD)

print("\nSimilarity range:")
print("Min:", cell_similarity.min())
print("Max:", cell_similarity.max())

print("\nNumber of undirected edges:",
      int(np.sum(np.triu(cell_adjacency, k=1))))

print("Non-zero adjacency entries:",
      int(np.sum(cell_adjacency)))

print("Density:",
      np.sum(cell_adjacency) / (n_cells * (n_cells - 1)))

# Basic symmetry check
if not np.allclose(cell_similarity, cell_similarity.T):
    raise ValueError("Similarity matrix is not symmetric.")

if not np.allclose(cell_adjacency, cell_adjacency.T):
    raise ValueError("Adjacency matrix is not symmetric.")

print("\n✓ CELL-CELL SIMILARITY GRAPH VERIFIED")

CELL-CELL SIMILARITY GRAPH
Similarity matrix shape: (28, 28)
Threshold: 0.75

Similarity range:
Min: 0.5510939
Max: 1.0

Number of undirected edges: 241
Non-zero adjacency entries: 482
Density: 0.63756615

✓ CELL-CELL SIMILARITY GRAPH VERIFIED


In [ ]:
# Save the cell-cell similarity and adjacency matrices

cell_similarity_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_similarity_methylation.npy"
)

cell_adjacency_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "pancreatic28_cell_adjacency_methylation_threshold075.npy"
)

np.save(cell_similarity_path, cell_similarity)
np.save(cell_adjacency_path, cell_adjacency)

print("Saved cell similarity:")
print(cell_similarity_path)

print("\nSaved cell adjacency:")
print(cell_adjacency_path)

print("\nSimilarity shape:", cell_similarity.shape)
print("Adjacency shape:", cell_adjacency.shape)
print("Undirected edges:",
      int(np.sum(np.triu(cell_adjacency, k=1))))

print("\n✓ CELL-CELL GRAPH SAVED")

Saved cell similarity:
D:\Pancreatic_GraphTCDR\data\processed\pancreatic28_cell_similarity_methylation.npy

Saved cell adjacency:
D:\Pancreatic_GraphTCDR\data\processed\pancreatic28_cell_adjacency_methylation_threshold075.npy

Similarity shape: (28, 28)
Adjacency shape: (28, 28)
Undirected edges: 241

✓ CELL-CELL GRAPH SAVED


In [ ]:
# Load and verify the 1,228 × 2,048 drug fingerprint matrix

drug_fp_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "drug_morgan_fingerprints_1228x2048.npy"
)

drug_fingerprints = np.load(drug_fp_path)

print("=" * 70)
print("DRUG MOLECULAR FINGERPRINTS")
print("=" * 70)

print("Shape:", drug_fingerprints.shape)
print("Expected:", (1228, 2048))
print("NaN:", np.isnan(drug_fingerprints).sum())
print("Inf:", np.isinf(drug_fingerprints).sum())

if drug_fingerprints.shape != (1228, 2048):
    raise ValueError("Unexpected fingerprint matrix shape.")

if not np.isfinite(drug_fingerprints).all():
    raise ValueError("Fingerprint matrix contains NaN or Inf.")

print("\n✓ DRUG FINGERPRINTS VERIFIED")

DRUG MOLECULAR FINGERPRINTS
Shape: (1228, 2048)
Expected: (1228, 2048)
NaN: 0
Inf: 0

✓ DRUG FINGERPRINTS VERIFIED


In [ ]:
# Construct the drug-drug similarity graph
# GraphTCDR-style threshold: |Pearson correlation| >= 0.75

from scipy.stats import pearsonr

n_drugs = drug_fingerprints.shape[0]

drug_similarity = np.zeros(
    (n_drugs, n_drugs),
    dtype=np.float32
)

for i in range(n_drugs):
    for j in range(i, n_drugs):

        r, _ = pearsonr(
            drug_fingerprints[i],
            drug_fingerprints[j]
        )

        drug_similarity[i, j] = r
        drug_similarity[j, i] = r

DRUG_THRESHOLD = 0.75

drug_adjacency = (
    np.abs(drug_similarity) >= DRUG_THRESHOLD
).astype(np.float32)

# Remove self-loops
np.fill_diagonal(drug_adjacency, 0)

print("=" * 70)
print("DRUG-DRUG SIMILARITY GRAPH")
print("=" * 70)

print("Similarity matrix shape:", drug_similarity.shape)
print("Threshold:", DRUG_THRESHOLD)

print("\nSimilarity range:")
print("Min:", drug_similarity.min())
print("Max:", drug_similarity.max())

undirected_edges = int(
    np.sum(np.triu(drug_adjacency, k=1))
)

print("\nNumber of undirected edges:", undirected_edges)

print(
    "Non-zero adjacency entries:",
    int(np.sum(drug_adjacency))
)

possible_edges = n_drugs * (n_drugs - 1)

print(
    "Directed adjacency density:",
    np.sum(drug_adjacency) / possible_edges
)

# Verification
if not np.allclose(
    drug_similarity,
    drug_similarity.T
):
    raise ValueError("Drug similarity matrix is not symmetric.")

if not np.allclose(
    drug_adjacency,
    drug_adjacency.T
):
    raise ValueError("Drug adjacency matrix is not symmetric.")

print("\n✓ DRUG-DRUG SIMILARITY GRAPH VERIFIED")

DRUG-DRUG SIMILARITY GRAPH
Similarity matrix shape: (1228, 1228)
Threshold: 0.75

Similarity range:
Min: -0.025009602
Max: 1.0

Number of undirected edges: 217
Non-zero adjacency entries: 434
Directed adjacency density: 0.000288036

✓ DRUG-DRUG SIMILARITY GRAPH VERIFIED


In [ ]:
# Save the drug-drug similarity and adjacency matrices

drug_similarity_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "drug_similarity_fingerprint_threshold075.npy"
)

drug_adjacency_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "drug_adjacency_fingerprint_threshold075.npy"
)

np.save(drug_similarity_path, drug_similarity)
np.save(drug_adjacency_path, drug_adjacency)

print("Saved drug similarity:")
print(drug_similarity_path)

print("\nSaved drug adjacency:")
print(drug_adjacency_path)

print("\nSimilarity shape:", drug_similarity.shape)
print("Adjacency shape:", drug_adjacency.shape)
print(
    "Undirected edges:",
    int(np.sum(np.triu(drug_adjacency, k=1)))
)

print("\n✓ DRUG-DRUG GRAPH SAVED")

Saved drug similarity:
D:\Pancreatic_GraphTCDR\data\processed\drug_similarity_fingerprint_threshold075.npy

Saved drug adjacency:
D:\Pancreatic_GraphTCDR\data\processed\drug_adjacency_fingerprint_threshold075.npy

Similarity shape: (1228, 1228)
Adjacency shape: (1228, 1228)
Undirected edges: 217

✓ DRUG-DRUG GRAPH SAVED


In [ ]:
# Verify the frozen cold-drug train/test split

import pandas as pd
import numpy as np
import os

table_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "integrated_modeling_table.csv"
)

modeling_df = pd.read_csv(table_path)

print("=" * 70)
print("VERIFY FROZEN COLD-DRUG SPLIT")
print("=" * 70)

print("Modeling table:", modeling_df.shape)

train_drugs = set(
    modeling_df.loc[
        modeling_df["drug_index"].isin(
            modeling_df["drug_index"].unique()
        ),
        "drug_index"
    ]
)

print("Total unique drugs:", modeling_df["drug_index"].nunique())

VERIFY FROZEN COLD-DRUG SPLIT
Modeling table: (18037, 7)
Total unique drugs: 1228


In [ ]:
import os

processed_dir = os.path.join(
    PROJECT,
    "data",
    "processed"
)

print("Files containing 'split' or 'train' or 'test':")

for filename in sorted(os.listdir(processed_dir)):
    name = filename.lower()

    if (
        "split" in name
        or "train" in name
        or "test" in name
    ):
        print(filename)

Files containing 'split' or 'train' or 'test':
splits


In [ ]:
splits_dir = os.path.join(
    PROJECT,
    "data",
    "processed",
    "splits"
)

print("Split folder:", splits_dir)
print("\nContents:")

for filename in sorted(os.listdir(splits_dir)):
    print(filename)

Split folder: D:\Pancreatic_GraphTCDR\data\processed\splits

Contents:
cold_drug_test.csv
cold_drug_train.csv


In [ ]:
train_path = os.path.join(
    splits_dir,
    "cold_drug_train.csv"
)

test_path = os.path.join(
    splits_dir,
    "cold_drug_test.csv"
)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_drugs = set(train_df["drug_index"])
test_drugs = set(test_df["drug_index"])

print("=" * 70)
print("FROZEN COLD-DRUG SPLIT VERIFICATION")
print("=" * 70)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nUnique train drugs:", len(train_drugs))
print("Unique test drugs:", len(test_drugs))

print("\nDrug overlap:", len(train_drugs & test_drugs))

print("\nTrain cells:", train_df["omics_index"].nunique())
print("Test cells:", test_df["omics_index"].nunique())

print("\nTrain response range:")
print(
    train_df["log_ic50"].min(),
    "to",
    train_df["log_ic50"].max()
)

print("\nTest response range:")
print(
    test_df["log_ic50"].min(),
    "to",
    test_df["log_ic50"].max()
)

if (
    train_df.shape[0] == 14235
    and test_df.shape[0] == 3802
    and len(train_drugs) == 982
    and len(test_drugs) == 246
    and len(train_drugs & test_drugs) == 0
):
    print("\n✓ FROZEN COLD-DRUG SPLIT VERIFIED")
else:
    print("\n⚠ SPLIT DOES NOT MATCH EXPECTED VALUES")

FROZEN COLD-DRUG SPLIT VERIFICATION
Train shape: (14235, 7)
Test shape: (3802, 7)

Unique train drugs: 982
Unique test drugs: 246

Drug overlap: 0

Train cells: 28
Test cells: 28

Train response range:
-77.69047248486987 to 287.7623388319992

Test response range:
-110.74656289456772 to 40.710561217845736

✓ FROZEN COLD-DRUG SPLIT VERIFIED


In [ ]:
print("=" * 70)
print("TRAINING DATA FOR RESPONSE-DERIVED GRAPH")
print("=" * 70)

print("Columns:")
print(train_df.columns.tolist())

print("\nShape:", train_df.shape)

print("\nFirst 5 rows:")
display(train_df.head())

print("\nUnique cells:", train_df["omics_index"].nunique())
print("Unique drugs:", train_df["drug_index"].nunique())

print("\nMissing values:")
print(train_df.isna().sum())

TRAINING DATA FOR RESPONSE-DERIVED GRAPH
Columns:
['depmap_id', 'ccle_name', 'omics_index', 'name', 'drug_index', 'log_ic50', 'r2']

Shape: (14235, 7)

First 5 rows:


,depmap_id,ccle_name,omics_index,name,drug_index,log_ic50,r2
0,ACH-000022,PATU8988S_PANCREAS,0,1-naphthyl-PP1,928,0.568421,-0.255038
1,ACH-000022,PATU8988S_PANCREAS,0,1-phenylbiguanide,1002,1.092774,-0.139651
2,ACH-000022,PATU8988S_PANCREAS,0,10-deacetylbaccatin,547,0.158021,0.385270
3,ACH-000022,PATU8988S_PANCREAS,0,10-hydroxycamptothecin,919,-1.413141,0.852317
4,ACH-000022,PATU8988S_PANCREAS,0,5-hydroxymethyl-tolterodine,1016,-0.201739,-0.500489



Unique cells: 28
Unique drugs: 982

Missing values:
depmap_id      0
ccle_name      0
omics_index    0
name           0
drug_index     0
log_ic50       0
r2             0
dtype: int64


In [ ]:
# Inspect the training response distribution by drug
# before constructing the response-derived cell-drug graph.

drug_response_summary = (
    train_df
    .groupby("drug_index")["log_ic50"]
    .agg(
        count="count",
        median="median",
        mean="mean",
        min="min",
        max="max"
    )
    .reset_index()
)

print("=" * 70)
print("TRAINING RESPONSE DISTRIBUTION BY DRUG")
print("=" * 70)

print("Number of drugs:", len(drug_response_summary))

print("\nPairs per drug:")
print(drug_response_summary["count"].describe())

print("\nMedian log(IC50) across drugs:")
print(drug_response_summary["median"].describe())

print("\nFirst 10 drugs:")
display(drug_response_summary.head(10))

TRAINING RESPONSE DISTRIBUTION BY DRUG
Number of drugs: 982

Pairs per drug:
count    982.000000
mean      14.495927
std        9.131918
min        1.000000
25%        6.000000
50%       15.000000
75%       23.000000
max       28.000000
Name: count, dtype: float64

Median log(IC50) across drugs:
count    982.000000
mean       0.339461
std        4.849029
min      -15.729489
25%        0.006881
50%        0.390910
75%        0.632043
max      144.529578
Name: median, dtype: float64

First 10 drugs:


,drug_index,count,median,mean,min,max
0,0,21,-1.169436,-1.075401,-2.103505,0.509513
1,1,28,-0.361841,-0.295161,-0.810261,0.664219
2,2,3,0.648061,0.648960,0.584440,0.714378
3,3,1,1.328124,1.328124,1.328124,1.328124
4,5,28,-2.049863,-1.828207,-2.582317,-0.779985
5,6,1,0.534565,0.534565,0.534565,0.534565
6,7,3,0.822875,0.834484,0.731158,0.949419
7,8,12,-0.233773,-0.140207,-0.498358,0.682372
8,9,28,-0.114053,-0.035877,-0.309422,0.818535
9,10,28,-0.119655,-0.142936,-0.893893,0.439044


In [ ]:
# Construct the response-derived cell-drug graph
# using TRAINING RESPONSES ONLY.
#
# GraphTCDR intended rule:
# For each drug, select the lower 50% of observed response values.
#
# Our adaptation:
# - use the frozen cold-drug TRAIN split only
# - use log10(IC50)
# - lower response = lower log10(IC50)
# - deterministic rank-based selection

import numpy as np
import pandas as pd
import torch
import os

response_graph_edges = []

drug_edge_summary = []

for drug_idx, group in train_df.groupby("drug_index", sort=True):

    group = group.sort_values(
        by=["log_ic50", "omics_index"],
        ascending=[True, True]
    ).reset_index(drop=True)

    n = len(group)

    # Number of lower-response observations to retain
    n_edges = int(np.ceil(n / 2))

    selected = group.iloc[:n_edges]

    for _, row in selected.iterrows():
        response_graph_edges.append([
            int(row["omics_index"]),
            int(row["drug_index"])
        ])

    drug_edge_summary.append({
        "drug_index": int(drug_idx),
        "training_pairs": n,
        "selected_edges": n_edges,
        "median_log_ic50": group["log_ic50"].median(),
        "threshold_log_ic50": group.iloc[n_edges - 1]["log_ic50"]
    })

response_graph_edges = np.asarray(
    response_graph_edges,
    dtype=np.int64
)

drug_edge_summary = pd.DataFrame(
    drug_edge_summary
)

print("=" * 70)
print("TRAINING-ONLY RESPONSE-DERIVED CELL-DRUG GRAPH")
print("=" * 70)

print("Training response pairs:", len(train_df))
print("Training drugs:", train_df["drug_index"].nunique())
print("Cell lines:", train_df["omics_index"].nunique())

print("\nResponse graph edges:", len(response_graph_edges))

print(
    "Unique cells in response graph:",
    np.unique(response_graph_edges[:, 0]).size
)

print(
    "Unique drugs in response graph:",
    np.unique(response_graph_edges[:, 1]).size
)

print("\nEdges per drug:")
print(drug_edge_summary["selected_edges"].describe())

print("\nFirst 10 drug summaries:")
display(drug_edge_summary.head(10))

# Verify every response-graph drug belongs to TRAIN
graph_drugs = set(response_graph_edges[:, 1])
train_drug_set = set(train_df["drug_index"])

if not graph_drugs.issubset(train_drug_set):
    raise ValueError(
        "LEAKAGE: response graph contains a drug outside the training set."
    )

# Verify no test drug entered the graph
test_drug_set = set(test_df["drug_index"])

if len(graph_drugs & test_drug_set) != 0:
    raise ValueError(
        "LEAKAGE: test drug found in response-derived graph."
    )

print("\nTrain/test drug overlap inside response graph:",
      len(graph_drugs & test_drug_set))

print("\n✓ TRAINING-ONLY RESPONSE GRAPH VERIFIED")

TRAINING-ONLY RESPONSE-DERIVED CELL-DRUG GRAPH
Training response pairs: 14235
Training drugs: 982
Cell lines: 28

Response graph edges: 7370
Unique cells in response graph: 28
Unique drugs in response graph: 982

Edges per drug:
count    982.000000
mean       7.505092
std        4.532977
min        1.000000
25%        3.000000
50%        8.000000
75%       12.000000
max       14.000000
Name: selected_edges, dtype: float64

First 10 drug summaries:


,drug_index,training_pairs,selected_edges,median_log_ic50,threshold_log_ic50
0,0,21,11,-1.169436,-1.169436
1,1,28,14,-0.361841,-0.363015
2,2,3,2,0.648061,0.648061
3,3,1,1,1.328124,1.328124
4,5,28,14,-2.049863,-2.058445
5,6,1,1,0.534565,0.534565
6,7,3,2,0.822875,0.822875
7,8,12,6,-0.233773,-0.248346
8,9,28,14,-0.114053,-0.128600
9,10,28,14,-0.119655,-0.133167



Train/test drug overlap inside response graph: 0

✓ TRAINING-ONLY RESPONSE GRAPH VERIFIED


In [ ]:
# Save the training-only response-derived cell-drug graph

response_edges_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "train_response_cell_drug_edges_lower50.csv"
)

response_summary_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "train_response_drug_edge_summary.csv"
)

edges_df = pd.DataFrame(
    response_graph_edges,
    columns=["omics_index", "drug_index"]
)

edges_df.to_csv(
    response_edges_path,
    index=False
)

drug_edge_summary.to_csv(
    response_summary_path,
    index=False
)

print("Saved response graph edges:")
print(response_edges_path)

print("\nShape:", edges_df.shape)

print("\nSaved drug edge summary:")
print(response_summary_path)

print("\n✓ RESPONSE GRAPH SAVED")

Saved response graph edges:
D:\Pancreatic_GraphTCDR\data\processed\train_response_cell_drug_edges_lower50.csv

Shape: (7370, 2)

Saved drug edge summary:
D:\Pancreatic_GraphTCDR\data\processed\train_response_drug_edge_summary.csv

✓ RESPONSE GRAPH SAVED


In [ ]:
response_edges_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "train_response_cell_drug_edges_lower50.csv"
)

response_summary_path = os.path.join(
    PROJECT,
    "data",
    "processed",
    "train_response_drug_edge_summary.csv"
)

edges_df = pd.DataFrame(
    response_graph_edges,
    columns=["omics_index", "drug_index"]
)

edges_df.to_csv(response_edges_path, index=False)
drug_edge_summary.to_csv(response_summary_path, index=False)

print("Saved response graph edges:")
print(response_edges_path)

print("\nShape:", edges_df.shape)

print("\nSaved drug edge summary:")
print(response_summary_path)

print("\n✓ RESPONSE GRAPH SAVED")

Saved response graph edges:
D:\Pancreatic_GraphTCDR\data\processed\train_response_cell_drug_edges_lower50.csv

Shape: (7370, 2)

Saved drug edge summary:
D:\Pancreatic_GraphTCDR\data\processed\train_response_drug_edge_summary.csv

✓ RESPONSE GRAPH SAVED


In [ ]:
print("=" * 70)
print("INSPECTING RESPONSE-GRAPH CELL INDICES")
print("=" * 70)

print("Unique omics_index values in train_df:")
print(
    sorted(train_df["omics_index"].unique())[:50]
)

print("\nNumber of unique omics_index values:",
      train_df["omics_index"].nunique())

print("\nPancreatic mapping:")
display(mapping_check)

print("\nResponse graph cell indices:")
print(
    sorted(np.unique(response_graph_edges[:, 0]))[:50]
)

INSPECTING RESPONSE-GRAPH CELL INDICES
Unique omics_index values in train_df:
[np.int64(0), np.int64(39), np.int64(47), np.int64(68), np.int64(79), np.int64(84), np.int64(85), np.int64(91), np.int64(103), np.int64(123), np.int64(135), np.int64(160), np.int64(165), np.int64(166), np.int64(179), np.int64(234), np.int64(237), np.int64(266), np.int64(291), np.int64(294), np.int64(313), np.int64(319), np.int64(321), np.int64(333), np.int64(362), np.int64(384), np.int64(386), np.int64(400)]

Number of unique omics_index values: 28

Pancreatic mapping:


,index,cellLineName
0,0,PATU8988S_PANCREAS
1,39,PANC0327_PANCREAS
2,47,PANC1_PANCREAS
3,68,PSN1_PANCREAS
4,79,SUIT2_PANCREAS
5,84,SNU213_PANCREAS
6,85,PANC0813_PANCREAS
7,91,DANG_PANCREAS
8,103,HUPT3_PANCREAS
9,123,CAPAN2_PANCREAS



Response graph cell indices:
[np.int64(0), np.int64(39), np.int64(47), np.int64(68), np.int64(79), np.int64(84), np.int64(85), np.int64(91), np.int64(103), np.int64(123), np.int64(135), np.int64(160), np.int64(165), np.int64(166), np.int64(179), np.int64(234), np.int64(237), np.int64(266), np.int64(291), np.int64(294), np.int64(313), np.int64(319), np.int64(321), np.int64(333), np.int64(362), np.int64(384), np.int64(386), np.int64(400)]


In [ ]:
# Convert original 402-cell indices to local pancreatic node indices 0–27

original_to_local = {
    int(row["index"]): local_idx
    for local_idx, (_, row) in enumerate(mapping_check.iterrows())
}

print("=" * 70)
print("402-CELL INDEX → 28-CELL NODE INDEX")
print("=" * 70)

print("Number of mappings:", len(original_to_local))

print("\nFirst 10 mappings:")
for original_idx, local_idx in list(original_to_local.items())[:10]:
    print(
        f"Original index {original_idx:3d}"
        f" -> Pancreatic node {local_idx:2d}"
    )

# Convert response graph cell indices
response_graph_edges_local = response_graph_edges.copy()

response_graph_edges_local[:, 0] = np.array([
    original_to_local[int(idx)]
    for idx in response_graph_edges[:, 0]
])

print("\nOriginal response edges:",
      len(response_graph_edges))

print("Local response edges:",
      len(response_graph_edges_local))

print(
    "Local cell indices:",
    response_graph_edges_local[:, 0].min(),
    "to",
    response_graph_edges_local[:, 0].max()
)

if (
    response_graph_edges_local[:, 0].min() >= 0
    and response_graph_edges_local[:, 0].max() < 28
):
    print("\n✓ CELL INDICES SUCCESSFULLY CONVERTED TO 0–27")
else:
    raise ValueError("Local cell-index conversion failed.")

402-CELL INDEX → 28-CELL NODE INDEX
Number of mappings: 28

First 10 mappings:
Original index   0 -> Pancreatic node  0
Original index  39 -> Pancreatic node  1
Original index  47 -> Pancreatic node  2
Original index  68 -> Pancreatic node  3
Original index  79 -> Pancreatic node  4
Original index  84 -> Pancreatic node  5
Original index  85 -> Pancreatic node  6
Original index  91 -> Pancreatic node  7
Original index 103 -> Pancreatic node  8
Original index 123 -> Pancreatic node  9

Original response edges: 7370
Local response edges: 7370
Local cell indices: 0 to 27

✓ CELL INDICES SUCCESSFULLY CONVERTED TO 0–27


In [ ]:
# Final verification of all heterogeneous graph indices

print("=" * 70)
print("FINAL HETEROGENEOUS GRAPH INDEX CHECK")
print("=" * 70)

# -------------------------
# Cell nodes
# -------------------------
print("CELL NODES")
print("Count:", 28)
print("Valid range: 0–27")
print(
    "Response graph range:",
    response_graph_edges_local[:, 0].min(),
    "to",
    response_graph_edges_local[:, 0].max()
)

# -------------------------
# Drug nodes
# -------------------------
print("\nDRUG NODES")
print("Count:", 1228)
print("Valid range: 0–1227")
print(
    "Response graph range:",
    response_graph_edges_local[:, 1].min(),
    "to",
    response_graph_edges_local[:, 1].max()
)

# -------------------------
# Cell-cell graph
# -------------------------
print("\nCELL-CELL GRAPH")
print("Shape:", cell_adjacency.shape)
print(
    "Edges:",
    int(np.sum(np.triu(cell_adjacency, k=1)))
)

# -------------------------
# Drug-drug graph
# -------------------------
print("\nDRUG-DRUG GRAPH")
print("Shape:", drug_adjacency.shape)
print(
    "Edges:",
    int(np.sum(np.triu(drug_adjacency, k=1)))
)

# -------------------------
# Response graph
# -------------------------
print("\nCELL-DRUG RESPONSE GRAPH")
print("Edges:", len(response_graph_edges_local))
print(
    "Unique cells:",
    np.unique(response_graph_edges_local[:, 0]).size
)
print(
    "Unique drugs:",
    np.unique(response_graph_edges_local[:, 1]).size
)

# -------------------------
# Validity checks
# -------------------------
if response_graph_edges_local[:, 0].min() < 0:
    raise ValueError("Invalid cell index.")

if response_graph_edges_local[:, 0].max() >= 28:
    raise ValueError("Cell index exceeds 27.")

if response_graph_edges_local[:, 1].min() < 0:
    raise ValueError("Invalid drug index.")

if response_graph_edges_local[:, 1].max() >= 1228:
    raise ValueError("Drug index exceeds 1227.")

if cell_adjacency.shape != (28, 28):
    raise ValueError("Invalid cell graph.")

if drug_adjacency.shape != (1228, 1228):
    raise ValueError("Invalid drug graph.")

print("\n✓ ALL HETEROGENEOUS GRAPH INDICES VERIFIED")

FINAL HETEROGENEOUS GRAPH INDEX CHECK
CELL NODES
Count: 28
Valid range: 0–27
Response graph range: 0 to 27

DRUG NODES
Count: 1228
Valid range: 0–1227
Response graph range: 0 to 1226

CELL-CELL GRAPH
Shape: (28, 28)
Edges: 241

DRUG-DRUG GRAPH
Shape: (1228, 1228)
Edges: 217

CELL-DRUG RESPONSE GRAPH
Edges: 7370
Unique cells: 28
Unique drugs: 982

✓ ALL HETEROGENEOUS GRAPH INDICES VERIFIED


In [ ]:
# Prepare heterogeneous graph node features

import torch
import numpy as np

# -------------------------
# Cell node features
# -------------------------
cell_x = torch.tensor(
    cell_features,
    dtype=torch.float32
)

# -------------------------
# Drug node features
# -------------------------
drug_x = torch.tensor(
    drug_fingerprints,
    dtype=torch.float32
)

print("=" * 70)
print("HETEROGENEOUS GRAPH NODE FEATURES")
print("=" * 70)

print("Cell features:")
print("  Shape:", tuple(cell_x.shape))
print("  Expected:", (28, 512))

print("\nDrug features:")
print("  Shape:", tuple(drug_x.shape))
print("  Expected:", (1228, 2048))

print("\nCell NaN:", torch.isnan(cell_x).sum().item())
print("Cell Inf:", torch.isinf(cell_x).sum().item())

print("\nDrug NaN:", torch.isnan(drug_x).sum().item())
print("Drug Inf:", torch.isinf(drug_x).sum().item())

if cell_x.shape != (28, 512):
    raise ValueError("Invalid cell feature shape.")

if drug_x.shape != (1228, 2048):
    raise ValueError("Invalid drug feature shape.")

if not torch.isfinite(cell_x).all():
    raise ValueError("Cell features contain NaN/Inf.")

if not torch.isfinite(drug_x).all():
    raise ValueError("Drug features contain NaN/Inf.")

print("\n✓ NODE FEATURES VERIFIED")

HETEROGENEOUS GRAPH NODE FEATURES
Cell features:
  Shape: (28, 512)
  Expected: (28, 512)

Drug features:
  Shape: (1228, 2048)
  Expected: (1228, 2048)

Cell NaN: 0
Cell Inf: 0

Drug NaN: 0
Drug Inf: 0

✓ NODE FEATURES VERIFIED


In [ ]:
# Reload previously saved graph components


import os
import numpy as np
import pandas as pd
import torch

PROJECT = r"D:\Pancreatic_GraphTCDR"
PROCESSED = os.path.join(PROJECT, "data", "processed")

# --------------------------------------------------
# 1. Load 28 × 512 pancreatic AE embeddings
# --------------------------------------------------
embedding_path = os.path.join(
    PROCESSED,
    "pancreatic28_ae_embeddings_512.csv"
)

ae_df = pd.read_csv(embedding_path)

cell_names = ae_df["cellLineName"].values
cell_features = ae_df.drop(
    columns=["cellLineName"]
).to_numpy(dtype=np.float32)

# --------------------------------------------------
# 2. Load 1228 × 2048 drug fingerprints
# --------------------------------------------------
drug_fp_path = os.path.join(
    PROCESSED,
    "drug_morgan_fingerprints_1228x2048.npy"
)

drug_fingerprints = np.load(drug_fp_path)

# --------------------------------------------------
# 3. Load cell–cell graph
# --------------------------------------------------
cell_similarity = np.load(
    os.path.join(
        PROCESSED,
        "pancreatic28_cell_similarity_methylation.npy"
    )
)

cell_adjacency = np.load(
    os.path.join(
        PROCESSED,
        "pancreatic28_cell_adjacency_methylation_threshold075.npy"
    )
)

# --------------------------------------------------
# 4. Load drug–drug graph
# --------------------------------------------------
drug_similarity = np.load(
    os.path.join(
        PROCESSED,
        "drug_similarity_fingerprint_threshold075.npy"
    )
)

drug_adjacency = np.load(
    os.path.join(
        PROCESSED,
        "drug_adjacency_fingerprint_threshold075.npy"
    )
)

# --------------------------------------------------
# 5. Load training-only response graph
# --------------------------------------------------
response_edges_path = os.path.join(
    PROCESSED,
    "train_response_cell_drug_edges_lower50.csv"
)

response_graph_edges = pd.read_csv(
    response_edges_path
).to_numpy(dtype=np.int64)

# --------------------------------------------------
# Summary
# --------------------------------------------------
print("=" * 70)
print("SAVED GRAPH COMPONENTS RELOADED")
print("=" * 70)

print("Cell features:", cell_features.shape)
print("Drug fingerprints:", drug_fingerprints.shape)

print("Cell similarity:", cell_similarity.shape)
print("Cell adjacency:", cell_adjacency.shape)

print("Drug similarity:", drug_similarity.shape)
print("Drug adjacency:", drug_adjacency.shape)

print("Response graph edges:", response_graph_edges.shape)

print("\n✓ ALL SAVED COMPONENTS RELOADED")

SAVED GRAPH COMPONENTS RELOADED
Cell features: (28, 512)
Drug fingerprints: (1228, 2048)
Cell similarity: (28, 28)
Cell adjacency: (28, 28)
Drug similarity: (1228, 1228)
Drug adjacency: (1228, 1228)
Response graph edges: (7370, 2)

✓ ALL SAVED COMPONENTS RELOADED


### Reconstruct the Frozen Cold-Drug Split and Cell-Index Mapping

The heterogeneous graph uses 28 pancreatic cancer cell lines, but the original
GraphTCDR multi-omics files contain 402 cell lines.

Therefore, two different cell indices exist:

- **Original 402-cell index:** the row index used by the GraphTCDR feature files.
- **Local 28-cell index:** the node index used in our pancreatic heterogeneous graph.

The frozen cold-drug split was created before graph construction and must remain
unchanged throughout the experiment.

We reload both the split and the pancreatic cell mapping so that all graph
components use consistent indices.

This is important because an incorrect index mapping could connect a response
from one cell line to the features of another cell line.

reload the frozen cold-drug spilt and pancreatic cell mapping

In [ ]:
# Reload the frozen cold-drug split and pancreatic cell mapping

import os
import pandas as pd
import numpy as np

PROJECT = r"D:\Pancreatic_GraphTCDR"
PROCESSED = os.path.join(PROJECT, "data", "processed")

# --------------------------------------------------
# Load frozen cold-drug split
# --------------------------------------------------

splits_dir = os.path.join(PROCESSED, "splits")

train_path = os.path.join(
    splits_dir,
    "cold_drug_train.csv"
)

test_path = os.path.join(
    splits_dir,
    "cold_drug_test.csv"
)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# --------------------------------------------------
# Load pancreatic cell mapping
# --------------------------------------------------

mapping_path = os.path.join(
    PROCESSED,
    "pancreatic28_cell_mapping.csv"
)

mapping = pd.read_csv(mapping_path)

# --------------------------------------------------
# Verify
# --------------------------------------------------

print("=" * 70)
print("FROZEN SPLIT + CELL MAPPING RELOADED")
print("=" * 70)

print("\nTRAINING SET")
print("Rows:", len(train_df))
print("Unique drugs:", train_df["drug_index"].nunique())
print("Unique cells:", train_df["omics_index"].nunique())

print("\nTEST SET")
print("Rows:", len(test_df))
print("Unique drugs:", test_df["drug_index"].nunique())
print("Unique cells:", test_df["omics_index"].nunique())

train_drugs = set(train_df["drug_index"])
test_drugs = set(test_df["drug_index"])

print("\nDrug overlap:", len(train_drugs & test_drugs))

print("\nPANCREATIC CELL MAPPING")
print("Shape:", mapping.shape)
print("Original 402-cell indices:", mapping["index"].tolist())

# --------------------------------------------------
# Safety checks
# --------------------------------------------------

if len(train_drugs & test_drugs) != 0:
    raise ValueError(
        "LEAKAGE ERROR: train and test drugs overlap."
    )

if mapping.shape[0] != 28:
    raise ValueError(
        "Expected exactly 28 pancreatic cell lines."
    )

if train_df["omics_index"].nunique() != 28:
    raise ValueError(
        "Training split does not contain all 28 pancreatic cells."
    )

if test_df["omics_index"].nunique() != 28:
    raise ValueError(
        "Test split does not contain all 28 pancreatic cells."
    )

print("\n✓ FROZEN SPLIT VERIFIED")
print("✓ 28-CELL MAPPING VERIFIED")
print("✓ NO TRAIN/TEST DRUG OVERLAP")